In [ ]:
import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import json

from src.config import PROCESSED_DATA_DIR, POSTERIOR_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model_pymc import build_and_fit, load_trace
from src.predict import predict_all, predict_match

In [ ]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates=["Date"])
team_mapping = json.load(open(PROCESSED_DATA_DIR / "team_mapping.json"))

train_df = df[df["season"].isin(TRAIN_SEASONS)].reset_index(drop=True)
test_df  = df[df["season"].isin(TEST_SEASONS)].reset_index(drop=True)

print(f"Training on {len(train_df)} matches, {len(team_mapping)} teams")
print(f"Testing on  {len(test_df)} matches")

In [ ]:
# Fit the model — takes ~10-20 min on CPU for 7 seasons
# To skip fitting and load a previously saved trace instead, comment out
# the line below and uncomment the one after it.
idata = build_and_fit(train_df, team_mapping, draws=1000, tune=500)
# idata = load_trace("trace.nc")

In [ ]:
# Summarise the global parameters and check convergence.
# r_hat measures whether the MCMC chains agree with each other —
# values close to 1.0 mean the chains converged; >= 1.01 is a warning sign.
summary = az.summary(idata, var_names=["mu", "home_adv", "sigma_att", "sigma_def"])
display(summary)

assert (summary["r_hat"] < 1.01).all(), "Convergence warning: r_hat >= 1.01 for some parameters"

In [ ]:
# Trace plots: left column shows the full chain (should look like fuzzy caterpillars),
# right column shows the marginal distribution of each parameter.
az.plot_trace(idata, var_names=["mu", "home_adv", "sigma_att", "sigma_def"])
plt.tight_layout()
plt.show()

In [ ]:
preds = predict_all(test_df, idata)
display(
    preds[["HomeTeam", "AwayTeam", "FTR", "p_home_win", "p_draw", "p_away_win", "mode_scoreline"]]
    .head(10)
)

In [ ]:
# Predict a single match by name — useful for interactive exploration.
result = predict_match("Arsenal", "Chelsea", idata, team_mapping)
print(result)

In [ ]:
import os
os.makedirs("outputs/tables", exist_ok=True)
preds.to_csv("outputs/tables/bayesian_preds.csv", index=False)
print("Saved outputs/tables/bayesian_preds.csv")